In [ ]:
import re
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DATA_DIR = '/Users/hayden/coderepos_mac_mini/mitsui_commodity/data/'
HOLDOUT_START = 1827
VAL_FRAC      = 0.15

# Feature lags
LAGS_POS     = [1, 5]         # past lags for ticker + macro features
LAGS_NEG     = []             # removed: caused lookahead (shift(-k) = future prices)
LAGS_AR      = [1, 5]         # AR lags for own-target features (always positive)
ROLL_WINDOWS = (5, 20)        # rolling mean / vol windows

# Model
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
SEQ_LEN       = 20
HIDDEN_SIZE   = 32
N_LAYERS      = 3
LR            = 1e-3
EPOCHS        = 100
BATCH_SIZE    = 64
PATIENCE      = 15
DROPOUT       = 0.2
WEIGHT_DECAY  = 1e-4
SPEARMAN_TEMP = 0.5

INCLUDE_MACRO = True

train        = pd.read_csv(f'{DATA_DIR}/train.csv')
train_labels = pd.read_csv(f'{DATA_DIR}/train_labels.csv')
target_pairs = pd.read_csv(f'{DATA_DIR}/target_pairs.csv')

train        = train.sort_values('date_id').reset_index(drop=True)
train_labels = train_labels.sort_values('date_id').reset_index(drop=True)
assert (train['date_id'].values == train_labels['date_id'].values).all()

target_cols = [c for c in train_labels.columns if c != 'date_id']
n_days = len(train)

print(f'train {train.shape}  labels {train_labels.shape}  pairs {target_pairs.shape}')
print(f'total days: {n_days}, {len(target_cols)} targets')
print(f'device: {DEVICE}')
print(f'LAGS_POS={LAGS_POS}  LAGS_NEG={LAGS_NEG}  ROLL_WINDOWS={ROLL_WINDOWS}')
print(f'hidden={HIDDEN_SIZE} layers={N_LAYERS} epochs={EPOCHS} patience={PATIENCE}')

train (1961, 558)  labels (1961, 425)  pairs (424, 3)
total days: 1961, 424 targets
device: cpu
LAGS_POS=[1, 5]  LAGS_NEG=[]  ROLL_WINDOWS=(5, 20)
hidden=32 layers=1 epochs=100 patience=15


In [16]:
def parse_pair(pair_str):
    s = pair_str.strip()
    tokens = re.split(r'\s*([+-])\s*', s)
    result, sign = [], 1
    if tokens[0]:
        result.append((sign, tokens[0]))
    i = 1
    while i < len(tokens):
        op, ticker = tokens[i], tokens[i + 1]
        result.append((1 if op == '+' else -1, ticker))
        i += 2
    return result

def get_exchange(ticker):
    if ticker.startswith('US_Stock_'):
        return 'US_STOCK'
    return ticker.split('_')[0]

rows = []
for _, r in target_pairs.iterrows():
    components = parse_pair(r['pair'])
    tickers = [t for _, t in components]
    exchanges = set(get_exchange(t) for t in tickers)
    rows.append({
        'target': r['target'],
        'lag': r['lag'],
        'pair_raw': r['pair'],
        'tickers': tickers,
        'exchange_key': '_'.join(sorted(exchanges)),
    })
target_info = pd.DataFrame(rows)
print(target_info.head())
print(f'\nUnique exchange keys: {sorted(target_info["exchange_key"].unique())}')
print(f'Lag distribution: {target_info["lag"].value_counts().to_dict()}')

     target  lag                                        pair_raw  \
0  target_0    1                           US_Stock_VT_adj_close   
1  target_1    1            LME_PB_Close - US_Stock_VT_adj_close   
2  target_2    1                     LME_CA_Close - LME_ZS_Close   
3  target_3    1                     LME_AH_Close - LME_ZS_Close   
4  target_4    1  LME_AH_Close - JPX_Gold_Standard_Futures_Close   

                                           tickers  exchange_key  
0                          [US_Stock_VT_adj_close]      US_STOCK  
1            [LME_PB_Close, US_Stock_VT_adj_close]  LME_US_STOCK  
2                     [LME_CA_Close, LME_ZS_Close]           LME  
3                     [LME_AH_Close, LME_ZS_Close]           LME  
4  [LME_AH_Close, JPX_Gold_Standard_Futures_Close]       JPX_LME  

Unique exchange keys: ['FX', 'FX_JPX', 'FX_LME', 'JPX_LME', 'JPX_US_STOCK', 'LME', 'LME_US_STOCK', 'US_STOCK']
Lag distribution: {1: 106, 2: 106, 3: 106, 4: 106}


In [17]:
def classify_ticker(t):
    fams = set()
    if 'JPX_Gold' in t:     fams.add('gold')
    if 'JPX_Platinum' in t: fams.add('platinum')
    if 'Rubber' in t:       fams.add('rubber')
    if 'LME_CA' in t:       fams.add('copper')
    if 'LME_AH' in t:       fams.add('aluminum')
    if 'LME_PB' in t:       fams.add('lead')
    if 'LME_ZS' in t:       fams.add('zinc')
    if any(s in t for s in ['_GLD_','_IAU_','_GDX_','_NEM_','_AEM_','_GOLD_','_KGC_','_FNV_','_WPM_','_SLV_']):
        fams.add('gold')
    if any(s in t for s in ['_FCX_','_SCCO_']):
        fams.add('copper')
    return fams

def families_for_tickers(tickers):
    fams = set()
    for t in tickers:
        fams |= classify_ticker(t)
    return fams

In [18]:
def lret(s):
    return np.log(s / s.shift(1))

def build_engineered(train, lags=LAGS_POS):
    out = {}
    fam_cols = {f: [] for f in ['gold', 'platinum', 'rubber', 'copper', 'aluminum', 'lead', 'zinc']}

    series = {
        'usdjpy_ret':     lret(train['FX_USDJPY']),
        'gld_ret':        lret(train['US_Stock_GLD_adj_close']),
        'gsr':            train['US_Stock_GLD_adj_close'] / train['US_Stock_SLV_adj_close'].replace(0, np.nan),
        'plat_gold':      train['JPX_Platinum_Standard_Futures_Close'] / train['JPX_Gold_Standard_Futures_Close'].replace(0, np.nan),
        'xle_ret':        lret(train['US_Stock_XLE_adj_close']),
        'fxi_ret':        lret(train['US_Stock_FXI_adj_close']),
        'copper_gold':    train['LME_CA_Close'] / train['JPX_Gold_Standard_Futures_Close'].replace(0, np.nan),
        'fcx_ret':        lret(train['US_Stock_FCX_adj_close']),
        'xlb_ret':        lret(train['US_Stock_XLB_adj_close']),
    }
    series['gsr_ret']         = lret(series['gsr'])
    series['plat_gold_ret']   = lret(series['plat_gold'])
    series['copper_gold_ret'] = lret(series['copper_gold'])

    membership = {
        'usdjpy_ret':       ['gold','platinum','rubber'],
        'gld_ret':          ['gold'],
        'gsr':              ['gold'],
        'gsr_ret':          ['gold'],
        'plat_gold':        ['platinum'],
        'plat_gold_ret':    ['platinum'],
        'xle_ret':          ['rubber'],
        'fxi_ret':          ['rubber','copper'],
        'copper_gold':      ['copper','aluminum','lead','zinc'],
        'copper_gold_ret':  ['copper','aluminum','lead','zinc'],
        'fcx_ret':          ['copper'],
        'xlb_ret':          ['aluminum','lead','zinc'],
    }

    for base, fams in membership.items():
        for k in lags:
            name = f'{base}_lag{k}'
            out[name] = series[base].shift(k)
            for f in fams:
                fam_cols[f].append(name)

    feats = pd.DataFrame(out).ffill()  # ffill only; NaN handled by preprocess
    return feats, fam_cols

eng_df, fam_cols = build_engineered(train)
print(f'eng_df shape: {eng_df.shape}')

eng_df shape: (1961, 24)


In [19]:
def own_target_features(target_name, train_labels, target_lag,
                        ar_lags=LAGS_AR, vol_windows=ROLL_WINDOWS):
    """Lagged returns + vol + rolling mean of target itself from train_labels."""
    y = train_labels[target_name]
    parts = []
    for k in ar_lags:
        shift = max(k, target_lag)
        parts.append(y.shift(shift).rename(f'own_lag{k}'))
    y_known = y.shift(target_lag)
    for w in vol_windows:
        parts.append(y_known.rolling(w).std().rename(f'own_vol{w}'))
        parts.append(y_known.rolling(w).mean().rename(f'own_mean{w}'))  # NEW vs v1
    if len(vol_windows) >= 2:
        s_v = y_known.rolling(vol_windows[0]).std()
        l_v = y_known.rolling(vol_windows[-1]).std()
        parts.append((s_v / (l_v + 1e-8)).rename('own_vol_ratio'))
    return pd.concat(parts, axis=1)  # NaN left for preprocess to handle

def _resolve_price_col(ticker, cols):
    cset = set(cols)
    for cand in (ticker, f'{ticker}_Close', f'{ticker}_adj_close', f'{ticker}_close'):
        if cand in cset:
            return cand
    for c in cols:
        if c.startswith(ticker):
            return c
    return None

def ticker_features(target_row, train,
                    pos_lags=LAGS_POS, neg_lags=LAGS_NEG,
                    roll_windows=ROLL_WINDOWS):
    """
    Per-ticker log-return features:
      pos_lags  : past data, shifted >= target_lag to avoid leakage
      neg_lags  : local future context; NaN at inference time, median-imputed
      roll_windows: rolling mean for trend context
    """
    target_lag = target_row['lag']
    parts = []
    missing = []
    for tk in target_row['tickers']:
        col = _resolve_price_col(tk, train.columns)
        if col is None:
            missing.append(tk)
            continue
        s = lret(train[col])
        for k in pos_lags:
            shift = max(k, target_lag)
            parts.append(s.shift(shift).rename(f'{tk}_ret_lag{k}'))
        for k in neg_lags:
            # k is negative: future values during training, NaN at inference
            parts.append(s.shift(k).rename(f'{tk}_ret_lag{k}'))
        for w in roll_windows:
            parts.append(s.rolling(w).mean().shift(target_lag).rename(f'{tk}_rmean{w}'))
    if not parts:
        return pd.DataFrame(index=train.index), missing
    return pd.concat(parts, axis=1).ffill(), missing  # ffill only; no fillna(0)

def features_for_target(target_row, eng_df, fam_cols, train_labels, train,
                        include_all_macro=False, include_macro=True):
    X_tk, missing = ticker_features(target_row, train)
    parts = [X_tk.reset_index(drop=True)]
    if include_macro or include_all_macro:
        if include_all_macro:
            X_cross = eng_df.copy()
        else:
            fams = families_for_tickers(target_row['tickers'])
            cols = sorted({c for f in fams for c in fam_cols.get(f, [])})
            X_cross = eng_df[cols] if cols else pd.DataFrame(index=eng_df.index)
        parts.append(X_cross.reset_index(drop=True))
    X_own = own_target_features(target_row['target'], train_labels, target_row['lag'])
    parts.append(X_own.reset_index(drop=True))
    return pd.concat(parts, axis=1), missing

In [20]:
def preprocess(X_train, *others):
    """
    1. Replace inf with NaN
    2. Median-impute NaN using training column medians
    3. Standardize using training mean/std
    """
    def to_clean(X):
        X = np.array(X, dtype=np.float32)
        X[~np.isfinite(X)] = np.nan
        return X

    X_tr = to_clean(X_train)
    col_median = np.nanmedian(X_tr, axis=0)
    col_median = np.where(np.isnan(col_median), 0.0, col_median)  # fallback for all-NaN cols

    def impute(X):
        X = to_clean(X).copy()
        for j in range(X.shape[1]):
            mask = np.isnan(X[:, j])
            if mask.any():
                X[mask, j] = col_median[j]
        return X

    X_tr_imp = impute(X_tr)
    mu  = X_tr_imp.mean(axis=0, keepdims=True)
    sig = X_tr_imp.std(axis=0, keepdims=True) + 1e-8
    return [(impute(arr) - mu) / sig for arr in (X_train, *others)]

In [21]:
target_X = {}
target_y = {}
target_cols_per_target = {}
all_missing = {}

for _, tr in target_info.iterrows():
    X, missing = features_for_target(tr, eng_df, fam_cols, train_labels, train,
                                     include_macro=INCLUDE_MACRO)
    target_X[tr['target']] = X.values.astype(np.float32)
    target_y[tr['target']] = train_labels[tr['target']].values.astype(np.float32)
    target_cols_per_target[tr['target']] = X.columns.tolist()
    if missing:
        all_missing[tr['target']] = missing

n_feat_dist = pd.Series([v.shape[1] for v in target_X.values()]).value_counts().sort_index()
print(f'built features for {len(target_X)} targets  (include_macro={INCLUDE_MACRO})')
print(f'feature count distribution:\n{n_feat_dist}')
if all_missing:
    print(f'\n{len(all_missing)} targets had unresolved tickers:')
    for t, m in list(all_missing.items())[:3]:
        print(f'  {t}: {m}')

# spot-check
for t in ['target_0', 'target_2']:
    if t in target_cols_per_target:
        cols = target_cols_per_target[t]
        print(f'\n{t}: {len(cols)} features')
        print(f'  {cols}')

built features for 424 targets  (include_macro=True)
feature count distribution:
11      4
21    253
23    128
25      9
27      6
29     18
31      6
Name: count, dtype: int64

target_0: 11 features
  ['US_Stock_VT_adj_close_ret_lag1', 'US_Stock_VT_adj_close_ret_lag5', 'US_Stock_VT_adj_close_rmean5', 'US_Stock_VT_adj_close_rmean20', 'own_lag1', 'own_lag5', 'own_vol5', 'own_mean5', 'own_vol20', 'own_mean20', 'own_vol_ratio']

target_2: 25 features
  ['LME_CA_Close_ret_lag1', 'LME_CA_Close_ret_lag5', 'LME_CA_Close_rmean5', 'LME_CA_Close_rmean20', 'LME_ZS_Close_ret_lag1', 'LME_ZS_Close_ret_lag5', 'LME_ZS_Close_rmean5', 'LME_ZS_Close_rmean20', 'copper_gold_lag1', 'copper_gold_lag5', 'copper_gold_ret_lag1', 'copper_gold_ret_lag5', 'fcx_ret_lag1', 'fcx_ret_lag5', 'fxi_ret_lag1', 'fxi_ret_lag5', 'xlb_ret_lag1', 'xlb_ret_lag5', 'own_lag1', 'own_lag5', 'own_vol5', 'own_mean5', 'own_vol20', 'own_mean20', 'own_vol_ratio']


In [22]:
def mitsui_metric(y_true, y_pred, verbose=False):
    daily_corrs = []
    for i in range(len(y_true)):
        mask = ~np.isnan(y_true[i])
        if mask.sum() < 2:
            continue
        corr, _ = spearmanr(y_true[i, mask], y_pred[i, mask])
        if not np.isnan(corr):
            daily_corrs.append(corr)
    arr = np.array(daily_corrs)
    std = arr.std()
    if std == 0:
        raise ZeroDivisionError('std of daily correlations is zero')
    sharpe = arr.mean() / std
    if verbose:
        print(f'mean {arr.mean():.4f}, std {std:.4f}, sharpe {sharpe:.4f} ndays {len(arr)}')
    return sharpe

In [23]:
train_mask   = train['date_id'] < HOLDOUT_START
holdout_mask = train['date_id'] >= HOLDOUT_START

n_train   = int(train_mask.sum())
split_idx = int(n_train * (1 - VAL_FRAC))
n_holdout = int(holdout_mask.sum())

y_hold_true = train_labels.loc[holdout_mask, target_cols].values
print(f'n_train={n_train}, split_idx={split_idx}, holdout={n_holdout}  y_hold_true={y_hold_true.shape}')

n_train=1827, split_idx=1552, holdout=134  y_hold_true=(134, 424)


In [24]:
class SeqDataset(Dataset):
    def __init__(self, X, y, seq_len):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)
        self.seq_len = seq_len
        self.valid = [i for i in range(len(X) - seq_len)
                      if not np.isnan(y[i + seq_len])]
    def __len__(self):
        return len(self.valid)
    def __getitem__(self, idx):
        i = self.valid[idx]
        return self.X[i:i + self.seq_len], self.y[i + self.seq_len]

class LSTMRegressor(nn.Module):
    def __init__(self, n_feat, hidden=HIDDEN_SIZE, n_layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(n_feat, hidden, n_layers, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(self.dropout(out[:, -1, :])).squeeze(-1)

def soft_rank(x, temp=SPEARMAN_TEMP):
    """Differentiable soft rank via pairwise sigmoid. O(n^2) — fine for batch_size=64."""
    diff = x.unsqueeze(0) - x.unsqueeze(1)   # (n, n)
    return torch.sigmoid(diff / temp).sum(dim=1)

def soft_spearman_loss(pred, target, temp=SPEARMAN_TEMP):
    """1 - differentiable Spearman correlation. Minimizing drives Spearman toward 1."""
    rp = soft_rank(pred, temp)
    with torch.no_grad():
        rt = soft_rank(target, temp)          # target ranks are constants
    rp = rp - rp.mean()
    rt = rt - rt.mean()
    cos = (rp * rt).sum() / (rp.norm() * rt.norm() + 1e-8)
    return 1.0 - cos

print(f'LSTM hidden={HIDDEN_SIZE} layers={N_LAYERS}')
print(f'Loss: soft Spearman (temp={SPEARMAN_TEMP})')
print(f'Early stop: val Spearman, patience={PATIENCE}')

LSTM hidden=32 layers=1
Loss: soft Spearman (temp=0.5)
Early stop: val Spearman, patience=15


In [25]:
def train_one_target(target_name, verbose=True, seed=0):
    torch.manual_seed(seed)
    tr_row = target_info[target_info['target'] == target_name].iloc[0]
    X_full = target_X[target_name]
    y_full = target_y[target_name]
    n_feat = X_full.shape[1]

    if verbose:
        cols = target_cols_per_target[target_name]
        print(f'\n=== {target_name}  pair="{tr_row["pair_raw"]}"  lag={tr_row["lag"]} ===')
        print(f'  tickers: {tr_row["tickers"]}  families: {families_for_tickers(tr_row["tickers"])}')
        print(f'  features ({n_feat}): {cols}')

    X_tr, X_val, X_ho = preprocess(
        X_full[:split_idx],
        X_full[split_idx:n_train],
        X_full[n_train:],
    )
    y_tr  = y_full[:split_idx]
    y_val = y_full[split_idx:n_train]
    y_ho  = y_full[n_train:]

    train_ds = SeqDataset(X_tr,  y_tr,  SEQ_LEN)
    val_ds   = SeqDataset(X_val, y_val, SEQ_LEN)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

    if verbose:
        print(f'  windows: train={len(train_ds)}, val={len(val_ds)}')

    model = LSTMRegressor(n_feat).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_spearman = -np.inf
    best_state    = None
    patience_ctr  = 0
    stopped_epoch = EPOCHS

    for epoch in range(EPOCHS):
        # train
        model.train()
        tr_loss = 0.0
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = soft_spearman_loss(model(xb), yb)
            loss.backward()
            opt.step()
            tr_loss += loss.item() * len(yb)
        tr_loss /= max(len(train_ds), 1)

        # validate
        model.eval()
        vp, vt = [], []
        with torch.no_grad():
            for xb, yb in val_dl:
                vp.extend(model(xb.to(DEVICE)).cpu().numpy())
                vt.extend(yb.numpy())

        val_sp = spearmanr(vt, vp)[0] if len(vp) >= 2 else -1.0
        if np.isnan(val_sp):
            val_sp = -1.0

        # early stop on val Spearman
        if val_sp > best_spearman:
            best_spearman = val_sp
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                stopped_epoch = epoch + 1
                break

        if verbose and (epoch == 0 or (epoch + 1) % 20 == 0 or patience_ctr == PATIENCE - 1):
            print(f'  ep {epoch+1:>3}/{EPOCHS}  loss={tr_loss:.5f}  val_sp={val_sp:.4f}  best={best_spearman:.4f}  p={patience_ctr}')

    if best_state is not None:
        model.load_state_dict(best_state)
    if verbose:
        print(f'  stopped ep {stopped_epoch}, best val_sp={best_spearman:.4f}')

    # holdout inference
    X_hist = np.concatenate([X_tr, X_val, X_ho], axis=0)
    preds  = np.zeros(len(y_ho), dtype=np.float32)
    model.eval()
    with torch.no_grad():
        for j in range(len(y_ho)):
            end = n_train + j
            window = X_hist[end - SEQ_LEN:end]
            preds[j] = model(torch.from_numpy(window).unsqueeze(0).to(DEVICE)).item()
    return preds

# demo
demo_pred = train_one_target('target_0', verbose=True)


=== target_0  pair="US_Stock_VT_adj_close"  lag=1 ===
  tickers: ['US_Stock_VT_adj_close']  families: set()
  features (11): ['US_Stock_VT_adj_close_ret_lag1', 'US_Stock_VT_adj_close_ret_lag5', 'US_Stock_VT_adj_close_rmean5', 'US_Stock_VT_adj_close_rmean20', 'own_lag1', 'own_lag5', 'own_vol5', 'own_mean5', 'own_vol20', 'own_mean20', 'own_vol_ratio']
  windows: train=1432, val=236
  ep   1/100  loss=1.00938  val_sp=0.1214  best=0.1214  p=0
  ep  19/100  loss=0.69055  val_sp=0.0490  best=0.1399  p=14
  stopped ep 20, best val_sp=0.1399


In [26]:
predictions = np.full_like(y_hold_true, np.nan, dtype=np.float32)
for ti, t in enumerate(target_cols):
    p = train_one_target(t, verbose=(ti < 5))
    predictions[:, ti] = p
    if (ti + 1) % 25 == 0:
        print(f'  done {ti+1}/{len(target_cols)}')
print(f'predictions shape: {predictions.shape}, NaN count: {np.isnan(predictions).sum()}')


=== target_0  pair="US_Stock_VT_adj_close"  lag=1 ===
  tickers: ['US_Stock_VT_adj_close']  families: set()
  features (11): ['US_Stock_VT_adj_close_ret_lag1', 'US_Stock_VT_adj_close_ret_lag5', 'US_Stock_VT_adj_close_rmean5', 'US_Stock_VT_adj_close_rmean20', 'own_lag1', 'own_lag5', 'own_vol5', 'own_mean5', 'own_vol20', 'own_mean20', 'own_vol_ratio']
  windows: train=1432, val=236
  ep   1/100  loss=1.00938  val_sp=0.1214  best=0.1214  p=0
  ep  19/100  loss=0.69055  val_sp=0.0490  best=0.1399  p=14
  stopped ep 20, best val_sp=0.1399

=== target_1  pair="LME_PB_Close - US_Stock_VT_adj_close"  lag=1 ===
  tickers: ['LME_PB_Close', 'US_Stock_VT_adj_close']  families: {'lead'}
  features (21): ['LME_PB_Close_ret_lag1', 'LME_PB_Close_ret_lag5', 'LME_PB_Close_rmean5', 'LME_PB_Close_rmean20', 'US_Stock_VT_adj_close_ret_lag1', 'US_Stock_VT_adj_close_ret_lag5', 'US_Stock_VT_adj_close_rmean5', 'US_Stock_VT_adj_close_rmean20', 'copper_gold_lag1', 'copper_gold_lag5', 'copper_gold_ret_lag1', 'cop

In [27]:
print('HOLDOUT SCORE')
score = mitsui_metric(y_hold_true, predictions, verbose=True)
print(f'Sharpe: {score:.4f}')

HOLDOUT SCORE
mean 0.0026, std 0.1317, sharpe 0.0200 ndays 134
Sharpe: 0.0200


In [28]:
print('=== Per-group Sharpe ===')
for ekey in sorted(target_info['exchange_key'].unique()):
    group_targets = target_info[target_info['exchange_key'] == ekey]['target'].tolist()
    y_idx  = [target_cols.index(t) for t in group_targets]
    g_true = y_hold_true[:, y_idx]
    g_pred = predictions[:, y_idx]
    try:
        s = mitsui_metric(g_true, g_pred)
        print(f'  {ekey:<20} {s:+.4f}  ({len(group_targets)} targets)')
    except ZeroDivisionError as e:
        print(f'  {ekey:<20} ERROR: {e}')

=== Per-group Sharpe ===
  FX                   -0.0899  (2 targets)
  FX_JPX               +0.0162  (46 targets)
  FX_LME               +0.0064  (86 targets)
  JPX_LME              -0.2156  (12 targets)
  JPX_US_STOCK         +0.1982  (87 targets)
  LME                  -0.1063  (8 targets)
  LME_US_STOCK         -0.0756  (181 targets)
  US_STOCK             -0.0851  (2 targets)
